# v107_tight_rule — v104's two-stage matcher, rule tuned on the tight (leaderboard-calibrated) mock

| Field | Value |
|---|---|
| **Version** | `v107_tight_rule` |
| **Plan group** | E2 / E5 (decision layer) |
| **Parent version** | v104 |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

v103's upload scored 0.961 public (+0.006 over v101) where the mock predicted +0.0027: the
test punishes false merges harder than the mock does. Both uploads share v101's matcher, so
their difference is the rule alone. Splitting each mock loss into its false-merge part L_FP
and its missed-match part L_FN, **public = 1 − L_FN − 1.45·L_FP − 0.0072** fits both points
(`mock.FP_WEIGHT`, `mock.PUBLIC_OFFSET`): the *tight mock*. This version keeps v104's
two-stage matcher and re-tunes the rule to maximise the tight score; `est_public` estimates
the leaderboard.

## 1. Hypothesis

* **Change vs parent (v104):** the rule only: tuned with `fp_weight=1.45` on the same mock
  tune entities (threshold grid, and expected-F0.5 decoding as a candidate). Stage 1, filter,
  stage 2 and every cached output are v104's.
* **Why:** a rule tuned for plain mock F0.5 keeps pairs whose false-merge risk the test
  prices 45 % higher; the tight tuning moves the thresholds to where the leaderboard's optimum
  should be.
* **Expected effect:** a stricter rule, plain mock F0.5 slightly down, est_public up.
* **Discard if:** est_public does not beat v104's by more than 0.001.

## 2. Setup

v104's configuration and artifacts (stage 1 = v101, stage-2 models, filter settings, the
stage-1 cache of the test partitions), plus the tight-mock constants.

In [1]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.data import isin
from entity_resolution.decision import apply_rule, decide, tune_expected
from entity_resolution.evaluate import error_samples
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.mock import FP_WEIGHT, PUBLIC_OFFSET, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import PipelineConfig, mock_scores, peak_rss_gb, tune_mock
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import TwoStage, run_test_two_stage

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v107_tight_rule"
ARTIFACTS = EXP_DIR / "artifacts"
PARENT = C.EXPERIMENTS / "v104_two_stage"
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000))          # v104's stage-1 config
v104 = TwoStage.load(PARENT / "artifacts", cfg)
STAGE1_CACHE = (cfg.cache_dir / "stage1" / f"v101_{cfg.blocking.key()}_f{v104.tcfg.floor}"
                f"_k{v104.tcfg.max_cands}_a{int(v104.tcfg.anchors)}")
parent = json.loads((PARENT / "metrics.json").read_text())
timings: dict[str, float] = {}
t_start = time.time()
print("FP_WEIGHT", FP_WEIGHT, "PUBLIC_OFFSET", PUBLIC_OFFSET, "| v104 rule", v104.rule,
      "| v104 mock F0.5", parent["mock_f05"])

FP_WEIGHT 1.45 PUBLIC_OFFSET 0.0072 | v104 rule DecisionRule(tau_abs=0.68, tau_rel=0.7, tau_single=0.68, max_matches=11, one_to_one=True) | v104 mock F0.5 0.9744


## 3. Data

The mock fold (as in v103–v106) and v104's stage-2 scores of its tune and val entities
(`artifacts/mock_scored.parquet`, after the 1-to-1 across every present entity).

In [2]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID],
                      sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID], target_shape())
    del train, val, fit_fold, tune_fold
    scored = pd.read_parquet(PARENT / "artifacts" / "mock_scored.parquet")
f"{len(scored):,} scored tune + val pairs"

'2,527,324 scored tune + val pairs'

## 4. Method

The threshold grid tuned for the tight score, and expected-F0.5 decoding tuned the same way;
the better of the two on the tune entities (tight score) is the rule.

In [3]:
t0 = time.time()
rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
tune_part = mock.part("tune")
rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                gammas=(0.7, 0.85, 1.0, 1.2, 1.5, 2.0),
                                misses=(0.0, 0.05, 0.1, 0.2, 0.4), fp_weight=FP_WEIGHT)
best_t, best_e = table_t["f_beta"].max(), table_e["f_beta"].max()
rule = rule_e if best_e > best_t else rule_t
timings["tune_seconds"] = round(time.time() - t0, 2)
print(f"tight tune score: threshold {best_t:.5f} {rule_t}\n                  expected  {best_e:.5f} {rule_e}")
print("chosen:", rule)

tight tune score: threshold 0.97310 DecisionRule(tau_abs=0.73, tau_rel=0.8, tau_single=0.73, max_matches=11, one_to_one=True)
                  expected  0.97311 ExpectedRule(gamma=1.5, miss=0.05, max_matches=11, one_to_one=True)
chosen: ExpectedRule(gamma=1.5, miss=0.05, max_matches=11, one_to_one=True)


## 5. Evaluation

Plain mock F0.5, tight score and est_public on the mock val entities for v103 (v101 + mock
rule, public 0.961), v104 (plain-tuned rule) and v107 (tight-tuned rule).

In [4]:
cols = ["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision", "pair_recall"]
v103_scored = pd.read_parquet(C.EXPERIMENTS / "v103_mock_rule" / "artifacts" / "mock_scored.parquet")
v103_rule = json.loads((C.EXPERIMENTS / "v103_mock_rule" / "metrics.json").read_text())["metrics"]["rule"]
from entity_resolution.decision import DecisionRule
res = {"v103 (public 0.961)": mock_scores(v103_scored, mock, DecisionRule(**v103_rule)).loc["all", cols],
       "v104 plain rule": mock_scores(scored, mock, v104.rule).loc["all", cols],
       "v107 tight rule": mock_scores(scored, mock, rule).loc["all", cols]}
del v103_scored
by_country = mock_scores(scored, mock, rule)
table = pd.DataFrame(res).T
display(table.round(4))
by_country[cols].round(4)

,f_beta,f_tight,est_public,f_beta_singletons,pair_precision,pair_recall
v103 (public 0.961),0.9704,0.9682,0.9610,0.9784,0.9939,0.9296
v104 plain rule,0.9744,0.9726,0.9654,0.9868,0.9950,0.9390
v107 tight rule,0.9745,0.9731,0.9659,0.9837,0.9965,0.9342


,f_beta,f_tight,est_public,f_beta_singletons,pair_precision,pair_recall
all,0.9745,0.9731,0.9659,0.9837,0.9965,0.9342
India,0.9701,0.9686,0.9614,0.9815,0.9963,0.9233
US,0.9792,0.9779,0.9707,0.9861,0.9967,0.9461


## 6. Error analysis

Error counts on the mock val entities under both rules.

In [5]:
part = mock.part("val")
rows_v = scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))]
counts = {}
for label, r in (("v104 plain rule", v104.rule), ("v107 tight rule", rule)):
    m = apply_rule(rows_v, r)
    counts[label] = {k: len(error_samples(m, part, k, n=10**9))
                     for k in ("false_merge", "missed", "false_singleton", "singleton_merge")}
pd.DataFrame(counts)

,v104 plain rule,v107 tight rule
false_merge,5265,3522
missed,68406,74223
false_singleton,3325,3154
singleton_merge,297,362


## 7. Log the result

In [6]:
ts = TwoStage(v104.stage1, v104.models, rule, v104.tcfg,
              table_e if rule is rule_e else table_t,
              {**v104.info, "retuned_from": asdict(v104.rule), "fp_weight": FP_WEIGHT})
ts.save(ARTIFACTS)
record = {
    "hypothesis": "tuning the rule for the leaderboard-calibrated tight mock raises est_public",
    "fp_weight": FP_WEIGHT, "public_offset": PUBLIC_OFFSET, "rule": asdict(rule),
    "rule_kind": type(rule).__name__, "parent_rule": asdict(v104.rule),
    "tight_tune_threshold": float(best_t), "tight_tune_expected": float(best_e),
    "comparison": {k: v.to_dict() for k, v in res.items()},
    "by_country": by_country[cols].to_dict("index"), "errors_mock": counts, **timings,
}
est, est_parent = res["v107 tight rule"]["est_public"], res["v104 plain rule"]["est_public"]
DECISION = "KEEP" if est > est_parent + 0.001 else "DROP"
record["decision"] = DECISION
print(f"est_public v104 {est_parent:.4f} -> v107 {est:.4f} {DECISION}")
row = log_result(
    EXP_DIR, change="v104 two-stage; rule re-tuned for the tight mock (false merges x1.45)",
    group="E2", mock_f05=res["v107 tight rule"]["f_beta"], cand_recall=None,
    notes=f"est_public {est:.4f} (v104 plain rule {est_parent:.4f}); f_tight "
          f"{res['v107 tight rule']['f_tight']:.4f}",
    metrics=record, owner="M1", parent="v104", decision=DECISION)
row

est_public v104 0.9654 -> v107 0.9659 DROP


{'version': 'v107',
 'date': '2026-09-26',
 'group': 'E2',
 'change': 'v104 two-stage; rule re-tuned for the tight mock (false merges x1.45)',
 'local_f05': '',
 'mock_f05': '0.9745',
 'cand_recall': '',
 'public_f05': '',
 'commit': '86da932',
 'notes': 'est_public 0.9659 (v104 plain rule 0.9654); f_tight 0.9731',
 'owner': 'M1',
 'parent': 'v104',
 'decision': 'DROP'}

## 8. Conclusion

* Tuned for the tight mock, the rule becomes expected-F0.5 decoding (γ 1.5, expected misses
  0.05); the threshold grid's best tight score is within 1e-5 of it (τ 0.73 / rel 0.8).
* **est_public 0.9659** against 0.9654 for v104's plain-tuned rule (+0.0005: below the
  +0.001 KEEP bar, so DROP as a model change) and 0.9610 for v103 (public 0.961). False merges
  on the mock val entities 5.3k → 3.5k pairs, misses 68k → 74k: the trade the leaderboard
  priced at 1.45.
* **Uploaded as submission #4** anyway: it is the best public estimate available, and it is
  the first upload of the two-stage matcher (expected public ≈ 0.965–0.967).
* Test candidate file: 4.8 (India) – 6.0 (France) candidates per S1, 136 MB instead of 802 MB.


## 9. Test inference

v104's stage-1 test outputs come from the stage-1 cache (no stage-1 pass), stage 2 is v104's,
the rule is this version's. Files kept in `submissions/v107/`, then both validators.

In [7]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, cache_dir=STAGE1_CACHE / "test")
print(f"run_test {time.time() - t0:.0f} s")
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
n_s1 = s1n_test.groupby(C.COUNTRY).size()
by = test_matches[C.S1_ID].map(country_of)
display(pd.DataFrame({
    "s1": n_s1,
    "cands_per_s1": test_summary["n_cands"].groupby(test_summary.index.map(country_of)).sum() / n_s1,
    "matched_share": test_matches.groupby(by)[C.S1_ID].nunique() / n_s1,
    "matches_per_s1": test_matches.groupby(by).size() / n_s1}))
dest = C.ROOT / "submissions" / "v107"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

/home/suryaguru/StudioProjects/aws/business_entity_resolution/.venv/lib64/python3.12/site-packages/xgboost/core.py:774: UserWarning: [12:05:07] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


run_test 84 s


,s1,cands_per_s1,matched_share,matches_per_s1
France,259452,6.039637,0.945204,3.247556
India,809986,4.837877,0.936943,3.193335
US,663106,4.990585,0.940773,3.294078


PASS
 


ML Challenge 2026 — submission validator
  test dir: /home/suryaguru/StudioProjects/aws/business_entity_resolution/dataset/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (104566 empty, 1627978 non-empty).
  candidate_pairs.tsv: 1732544 rows (20777 empty, 1711767 non-empty).

PASS — no blocking issues found. Safe to submit.
 
notebook total 300 s, peak RSS 4.58 GB
